In [ ]:
import json
import platform
from datetime import datetime, timezone

environment = {
    "run_id": "colab_environment_check",
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "pytorch_version": None,
    "pytorch_cuda_version": None,
    "cuda_available": False,
    "gpus": [],
    "gpu_check_passed": False,
    "diagnostics": [],
}

try:
    import torch

    environment["pytorch_version"] = str(torch.__version__)
    environment["pytorch_cuda_version"] = torch.version.cuda
    environment["cuda_available"] = torch.cuda.is_available()

    if environment["cuda_available"]:
        for index in range(torch.cuda.device_count()):
            gpu = torch.cuda.get_device_properties(index)
            environment["gpus"].append({
                "index": index,
                "name": gpu.name,
                "total_vram_gib": round(gpu.total_memory / 1024**3, 2),
            })

        environment["gpu_check_passed"] = bool(environment["gpus"])
    else:
        environment["diagnostics"].append(
            "Chưa truy cập được GPU. Hãy kiểm tra loại runtime."
        )

except (ImportError, RuntimeError, OSError) as error:
    environment["diagnostics"].append(str(error))

print(json.dumps(environment, ensure_ascii=False, indent=2))

In [ ]:
from pathlib import Path
from google.colab import files

if "environment" not in globals():
    raise RuntimeError("Lỗi: Chưa có dữ liệu môi trường.")

report_path = Path("/content/outputs/colab_environment_check/config.json")
report_path.parent.mkdir(parents=True, exist_ok=True)

report_path.write_text(
    json.dumps(environment, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

files.download(str(report_path))

In [ ]:
%pip install "transformers>=4.57,<5" datasets sentencepiece rouge-score pandas matplotlib gradio

In [ ]:
import importlib
import json
import platform
from importlib.metadata import version

packages = {
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "sentencepiece": "sentencepiece",
    "rouge-score": "rouge_score",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "gradio": "gradio",
}

library_versions = {}

for package_name, module_name in packages.items():
    importlib.import_module(module_name)
    library_versions[package_name] = version(package_name)

print(json.dumps(
    {
        "python_version": platform.python_version(),
        "library_versions": library_versions,
        "import_check_passed": True,
    },
    ensure_ascii=False,
    indent=2,
))

In [ ]:
import json
import torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

if not torch.cuda.is_available():
    raise RuntimeError("Lỗi: GPU chưa khả dụng để tải mô hình lên CUDA.")

model_id = "VietAI/vit5-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast=False,
)

sample_text = "Rửa sạch hạt, ngâm qua đêm rồi gieo vào đất ẩm."

token_ids = tokenizer(
    sample_text,
    return_tensors="pt",
)["input_ids"][0]

decoded_text = tokenizer.decode(
    token_ids.tolist(),
    skip_special_tokens=True,
)

model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
model = model.to("cuda:0")
model.eval()

model_device = next(model.parameters()).device

if model_device.type != "cuda":
    raise RuntimeError("Lỗi: Mô hình chưa được chuyển lên GPU.")

print(json.dumps(
    {
        "model_id": model_id,
        "model_class": type(model).__name__,
        "tokenizer_class": type(tokenizer).__name__,
        "parameter_count": sum(p.numel() for p in model.parameters()),
        "model_device": str(model_device),
        "sample_text": sample_text,
        "token_ids": token_ids.tolist(),
        "decoded_text": decoded_text,
        "model_loading_check_passed": True,
    },
    ensure_ascii=False,
    indent=2,
))